# Taller Deep Learning con PyTorch 2026
**Cuaderno de práctica** · Instituto Tecnológico de Mérida

**Antes de empezar**
1. Menú *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)*. Es indispensable para el día 2.
2. Ejecuta las celdas en orden (`Shift + Enter`).
3. Las celdas con `...` o `TODO` son para que las completes.
4. La explicación teórica está en la presentación; aquí se practica.

## La receta de entrenamiento
Todo el taller usa los mismos cinco pasos:

| 1. Datos | 2. Modelo | 3. Pérdida | 4. Optimizador | 5. Bucle de entrenamiento |
|---|---|---|---|---|
| tensores | `nn.Sequential` | `nn.CrossEntropyLoss` / `nn.MSELoss` | `optim.Adam` / `optim.SGD` | `zero_grad → forward → loss → backward → step` |

# DÍA 1

## 0. Configuración

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim

torch.manual_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Usando:", device)

## 1. Una neurona desde cero: estatura y peso
**Objetivo:** predecir el sexo (0 = mujer, 1 = hombre) a partir de la estatura y el peso con **una sola neurona programada a mano con NumPy**, sin PyTorch.

Seguimos la receta: datos, modelo, pérdida, optimizador y bucle.

### Paso 1 · Datos

In [ ]:
# DATOS DE EJEMPLO (inventados). Reemplázalos con los del grupo:
# una persona por posición en las tres listas.
estaturas = [157, 160, 163, 155, 168, 162, 158, 165, 159, 170, 161, 154, 166, 164, 167,
             172, 178, 168, 180, 175, 183, 170, 177, 174, 169, 181, 173, 176, 185, 171]   # cm
pesos = [52, 58, 55, 49, 64, 60, 54, 70, 50, 66, 57, 51, 62, 59, 68,
         70, 78, 65, 85, 72, 90, 68, 80, 75, 74, 82, 66, 77, 88, 79]                      # kg
sexos = [0] * 15 + [1] * 15                                                               # 0 = mujer, 1 = hombre

In [ ]:
estaturas = np.array(estaturas)
pesos = np.array(pesos)
sexos = np.array(sexos)

plt.scatter(estaturas[sexos == 0], pesos[sexos == 0], c='tab:blue', edgecolor='k', label='mujer (0)')
plt.scatter(estaturas[sexos == 1], pesos[sexos == 1], c='tab:red', edgecolor='k', label='hombre (1)')
plt.xlabel('estatura (cm)'); plt.ylabel('peso (kg)')
plt.legend(); plt.show()

**Pregunta:** si pudieras dibujar **una recta** para separar los dos grupos, ¿dónde la pondrías? ¿Quedaría algún punto del lado equivocado?

### Paso 2 · Modelo

In [ ]:
X = np.column_stack([estaturas, pesos])    # (n, 2): una fila por persona
y = sexos                                  # (n,)
print(X.shape, y.shape)

def sigmoide(z):
    return 1 / (1 + np.exp(-z))

### Pasos 3, 4 y 5 · Pérdida, gradientes y bucle
Completa el bucle. Recuerda que con sigmoide y entropía cruzada: `dz = y_hat - y`.

**Intento 1:** ejecútalo con `datos = X` (sin normalizar) y observa la pérdida. Luego sigue con la celda de normalización.

In [ ]:
datos = X          # <- primero sin normalizar; después cámbialo por X_norm y vuelve a ejecutar

w = np.zeros(2)
b = 0.0
lr = 0.1
n = len(y)
perdidas = []
historial = []     # guarda la recta en algunas épocas

for epoca in range(500):
    z = ...                 # TODO: suma ponderada
    y_hat = ...             # TODO: sigmoide

    loss = -np.mean(y * np.log(y_hat + 1e-9) + (1 - y) * np.log(1 - y_hat + 1e-9))

    dz = ...                # TODO
    dw = ...                # TODO
    db = ...                # TODO

    w = ...                 # TODO
    b = ...                 # TODO

    perdidas.append(loss)
    if (epoca + 1) in [1, 10, 50, 500]:
        historial.append((epoca + 1, w, b))

plt.plot(perdidas); plt.xlabel('época'); plt.ylabel('pérdida'); plt.show()
print("Pérdida final:", perdidas[-1])

### Intento 2: normalizando
Estandariza cada columna: $(x - \text{media}) / \text{desviación}$. Después cambia `datos = X` por `datos = X_norm` en la celda anterior y ejecútala de nuevo. ¿Qué cambió en la pérdida?

In [ ]:
media = X.mean(axis=0)
desv = X.std(axis=0)
X_norm = ...        # TODO

print(X_norm.mean(axis=0), X_norm.std(axis=0))    # esperado: ~0 y ~1

### Evaluación
1. Calcula la precisión de tu neurona y compárala con un modelo que siempre predice la clase más frecuente.
2. Predice para una persona nueva: 172 cm y 68 kg. Normaliza con la **misma** `media` y `desv`.

Ejecuta esta parte con la neurona entrenada con `X_norm`.

In [ ]:
y_hat = sigmoide(X_norm @ w + b)

y_pred = ...          # TODO: 1 si y_hat > 0.5, 0 si no
precision = ...       # TODO
trivial = ...         # TODO: precisión de predecir siempre la clase más frecuente
print(f"Precisión: {precision:.2%} | Modelo trivial: {trivial:.2%}")

nuevo = np.array([172, 68])
prob = ...            # TODO: normaliza 'nuevo' y calcula la probabilidad de clase 1
print(f"Probabilidad de clase 1: {prob:.2f}")

### Visualización: la recta que aprende
La frontera es donde $z = 0$, es decir, $w_1 x_1 + w_2 x_2 + b = 0$. Se grafican las rectas guardadas en `historial`.

In [ ]:
plt.scatter(estaturas[sexos == 0], pesos[sexos == 0], c='tab:blue', edgecolor='k', label='mujer (0)')
plt.scatter(estaturas[sexos == 1], pesos[sexos == 1], c='tab:red', edgecolor='k', label='hombre (1)')

x1 = np.linspace(estaturas.min() - 5, estaturas.max() + 5, 100)

for epoca_i, w_i, b_i in historial:
    # pasar la recta de unidades normalizadas a cm y kg
    w_o = w_i / desv
    b_o = b_i - (w_i * media / desv).sum()
    x2 = -(w_o[0] * x1 + b_o) / w_o[1]
    plt.plot(x1, x2, label=f"época {epoca_i}")

plt.ylim(pesos.min() - 5, pesos.max() + 5)
plt.xlabel('estatura (cm)'); plt.ylabel('peso (kg)')
plt.legend(); plt.show()

**Reflexión**
- ¿Qué pasó con la pérdida sin normalizar? ¿Y al normalizar?
- ¿Qué líneas de este bucle crees que reemplazan `loss.backward()` y `optimizador.step()` en PyTorch?
- ¿Qué personas quedan del lado equivocado de la recta? ¿Puede una sola neurona separarlas?

## 2. Tensores
Ejecuta el ejemplo y revisa las formas (`shape`) que imprime.

In [ ]:
tensor_1 = torch.tensor([0.1, 1, 0.9, 0.7, 0.3])
tensor_2 = torch.tensor([[0, 0.2, 0.4, 0.6],
                         [1, 0.8, 0.6, 0.4]]).to(device)
print(tensor_1.shape)
print(tensor_2.shape)
print(tensor_2.device)

**Mini-reto**
- Crea un tensor de ceros de forma `(3, 4)`, uno de unos de forma `(2, 4)` y uno aleatorio de forma `(3, 4)`.
- Suma `ceros` y `aleatorio`.
- ¿Qué pasa si intentas sumar `unos` con `ceros`? ¿Por qué?

In [ ]:
# TODO
ceros = ...
unos = ...
aleatorio = ...

suma = ...
print(suma)

## 3. Autograd
Ejecuta el ejemplo. Antes de correrlo, ¿qué valor esperas en `a.grad`?

In [ ]:
a = torch.ones((2, 2), requires_grad=True)
b = a + 5
c = b.mean()

print('b =', b, '\n')
print('c =', c, '\n')

c.backward()
print(a.grad)

**Mini-reto**
Calcula con autograd la derivada de $y = 3x^2 + 2x$ en $x = 2$ y verifica a mano el resultado.

In [ ]:
x = torch.tensor([2.0], requires_grad=True)
y = ...             # TODO
y.backward()
print(x.grad)       # ¿coincide con tu cálculo a mano?

## 4. Descenso de gradiente
Minimizamos $f(x) = (x-3)^2 + 5$ empezando en $x = 0$. La función `descenso` devuelve la trayectoria de $x$.

In [ ]:
def descenso(lr=0.1, itera=30, opt_cls=optim.SGD):
    x = torch.tensor([0.0], requires_grad=True)
    optimizador = opt_cls([x], lr=lr)
    historial = []
    for i in range(itera):
        optimizador.zero_grad()      # limpiar gradientes
        y = (x - 3)**2 + 5           # forward
        y.backward()                 # backward
        optimizador.step()           # actualizar x
        historial.append(x.item())
    return historial

for lr in [0.01, 0.1, 0.5, 1.1]:
    plt.plot(descenso(lr=lr), label=f"lr={lr}")
plt.axhline(3, color='gray', linestyle='--')
plt.ylim(-2, 8)
plt.xlabel('iteración'); plt.ylabel('x'); plt.legend(); plt.show()

**Reto**
- Describe qué ocurre con cada `lr`. ¿Cuál converge más rápido? ¿Cuál no converge?
- Compara SGD con Adam usando el mismo `lr`. ¿En qué se diferencian las trayectorias?

In [ ]:
trayectoria_adam = descenso(lr=..., opt_cls=optim.Adam)   # TODO: elige un lr

plt.plot(descenso(lr=0.1), label='SGD')
plt.plot(trayectoria_adam, label='Adam')
plt.axhline(3, color='gray', linestyle='--')
plt.xlabel('iteración'); plt.ylabel('x'); plt.legend(); plt.show()

## 5. Primera red: clasificar *moons*
Aplicamos la receta completa. Ejecuta cada paso y observa qué cambia en la salida.

### Paso 1 · Datos

In [ ]:
from sklearn.datasets import make_moons

X_np, y_np = make_moons(n_samples=400, noise=0.15, random_state=42)
X = torch.tensor(X_np, dtype=torch.float32)
y = torch.tensor(y_np, dtype=torch.long)

plt.scatter(X_np[:, 0], X_np[:, 1], c=y_np, cmap='coolwarm', edgecolor='k', s=20)
plt.title("Dataset moons"); plt.show()
print(X.shape, y.shape)

### Pasos 2, 3 y 4 · Modelo, pérdida y optimizador

In [ ]:
modelo_lunas = nn.Sequential(
    nn.Linear(2, 16),
    nn.ReLU(),
    nn.Linear(16, 16),
    nn.ReLU(),
    nn.Linear(16, 2)
)

criterio = nn.CrossEntropyLoss()
optimizador = optim.Adam(modelo_lunas.parameters(), lr=0.01)

### Paso 5 · Bucle de entrenamiento

In [ ]:
epocas = 500
perdidas = []
for epoca in range(epocas):
    optimizador.zero_grad()
    predicciones = modelo_lunas(X)
    loss = criterio(predicciones, y)
    loss.backward()
    optimizador.step()

    perdidas.append(loss.item())
    if (epoca + 1) % 100 == 0:
        print(f"Época {epoca+1} | Pérdida: {loss.item():.4f}")

plt.plot(perdidas); plt.xlabel('época'); plt.ylabel('pérdida'); plt.show()

### Evaluación
Calcula la precisión sobre los datos y observa la frontera de decisión.

In [ ]:
with torch.no_grad():
    pred = modelo_lunas(X).argmax(dim=1)

precision = ...      # TODO: fracción de aciertos
print(f"Precisión: {precision:.2%}")

In [ ]:
def graficar_frontera(modelo, X_np, y_np, titulo=""):
    x_min, x_max = X_np[:, 0].min() - 0.5, X_np[:, 0].max() + 0.5
    y_min, y_max = X_np[:, 1].min() - 0.5, X_np[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                         np.linspace(y_min, y_max, 200))
    malla = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32)
    with torch.no_grad():
        zz = modelo(malla).argmax(dim=1).numpy().reshape(xx.shape)
    plt.contourf(xx, yy, zz, alpha=0.3, cmap='coolwarm')
    plt.scatter(X_np[:, 0], X_np[:, 1], c=y_np, cmap='coolwarm', edgecolor='k', s=20)
    plt.title(titulo); plt.show()

graficar_frontera(modelo_lunas, X_np, y_np, "Frontera de decisión")

### Actividad 1 · Experimentos
Responde en una celda de texto cuando termines:
1. ¿Qué pasa con la pérdida si `lr` es 0.001? ¿Y si es 0.5?
2. ¿Qué pasa si quitas las `ReLU`? ¿Por qué la frontera ya no puede curvarse?

La función `entrenar_lunas` repite la receta para que experimentes rápido.

In [ ]:
def entrenar_lunas(modelo, lr=0.01, epocas=500):
    criterio = nn.CrossEntropyLoss()
    optimizador = optim.Adam(modelo.parameters(), lr=lr)
    perdidas = []
    for _ in range(epocas):
        optimizador.zero_grad()
        loss = criterio(modelo(X), y)
        loss.backward()
        optimizador.step()
        perdidas.append(loss.item())
    return perdidas

In [ ]:
# Experimento A: otro learning rate
modelo_A = nn.Sequential(nn.Linear(2, 16), nn.ReLU(),
                         nn.Linear(16, 16), nn.ReLU(),
                         nn.Linear(16, 2))
perdidas_A = entrenar_lunas(modelo_A, lr=...)      # TODO: prueba 0.001 y 0.5

# Experimento B: la misma red, pero sin activaciones
modelo_B = nn.Sequential(
    ...                                            # TODO
)
perdidas_B = entrenar_lunas(modelo_B)

plt.plot(perdidas, label='original')
plt.plot(perdidas_A, label='experimento A')
plt.plot(perdidas_B, label='experimento B')
plt.xlabel('época'); plt.ylabel('pérdida'); plt.legend(); plt.show()

graficar_frontera(modelo_B, X_np, y_np, "Experimento B")

## 6. Ejercicio: regresión con el dataset AI4I (UCI)
**Objetivo:** predecir el `Torque` de una máquina a partir de sus otras variables numéricas.

Ahora **tú construyes** los pasos 2, 3, 4 y 5 de la receta. Cambia respecto a *moons*: el problema es de *regresión*, no de clasificación.

### Paso 1 · Datos (ya está listo)

In [ ]:
!pip install -q ucimlrepo
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

dataset = fetch_ucirepo(id=601)
df = dataset.data.features
num = df.select_dtypes('number')            # descarta la columna categórica 'Type'
print(num.columns.tolist())
num.head()

In [ ]:
col_objetivo = [c for c in num.columns if 'Torque' in c][0]
col_vel = [c for c in num.columns if 'Rotational' in c][0]

display(num.corr())
plt.scatter(num[col_vel], num[col_objetivo], s=3, alpha=0.4)
plt.xlabel(col_vel); plt.ylabel(col_objetivo); plt.show()

In [ ]:
X_np_r = num.drop(columns=col_objetivo).values
y_np_r = num[[col_objetivo]].values

X_tr, X_te, y_tr, y_te = train_test_split(X_np_r, y_np_r, test_size=0.2, random_state=42)

sx, sy = StandardScaler(), StandardScaler()
X_tr, X_te = sx.fit_transform(X_tr), sx.transform(X_te)
y_tr, y_te = sy.fit_transform(y_tr), sy.transform(y_te)

X_tr, X_te, y_tr, y_te = [torch.tensor(a, dtype=torch.float32)
                          for a in (X_tr, X_te, y_tr, y_te)]
print(X_tr.shape, y_tr.shape, X_te.shape)

### Pasos 2, 3 y 4 · Modelo, pérdida y optimizador
Piensa antes de escribir:
- ¿Cuántas entradas tiene la red? ¿Cuántas salidas?
- ¿Lleva activación la última capa? ¿Por qué?
- ¿Qué función de pérdida corresponde a regresión?

In [ ]:
n_entradas = X_tr.shape[1]

modelo_reg = nn.Sequential(
    ...                    # TODO
)
criterio_reg = ...         # TODO
optimizador_reg = ...      # TODO

### Paso 5 · Bucle de entrenamiento
En cada época: entrena con los datos de entrenamiento y mide la pérdida en prueba **sin calcular gradientes** (`torch.no_grad()`). Guarda ambas pérdidas.

In [ ]:
epocas = 300
perdidas_tr, perdidas_te = [], []

for epoca in range(epocas):
    # TODO: zero_grad, forward, pérdida, backward, step
    ...

    # TODO: pérdida en prueba sin gradientes y guardar ambas en las listas
    ...

    if (epoca + 1) % 50 == 0:
        print(f"Época {epoca+1} | train: {perdidas_tr[-1]:.4f} | test: {perdidas_te[-1]:.4f}")

In [ ]:
plt.plot(perdidas_tr, label='entrenamiento')
plt.plot(perdidas_te, label='prueba')
plt.xlabel('época'); plt.ylabel('MSE'); plt.legend(); plt.show()

### Evaluación
Los datos están estandarizados (media 0, desviación 1). Un modelo que siempre predice la media tendría un MSE cercano a 1.
1. Calcula el MSE de tu red en prueba y el del modelo que siempre predice la media. ¿Lo supera tu red?
2. Convierte el error a unidades reales (Nm): `RMSE en Nm = sqrt(MSE) * sy.scale_[0]`.

In [ ]:
modelo_reg.eval()
with torch.no_grad():
    pred_te = modelo_reg(X_te)

mse_te = ...            # TODO: MSE de tu red en prueba
mse_baseline = ...      # TODO: MSE de un modelo que siempre predice la media (0 en datos estandarizados)
rmse_nm = ...           # TODO: RMSE en Nm

print(f"MSE red: {mse_te:.4f} | MSE baseline: {mse_baseline:.4f} | RMSE: {rmse_nm:.2f} Nm")

In [ ]:
real = sy.inverse_transform(y_te.numpy())
estimado = sy.inverse_transform(pred_te.numpy())
plt.scatter(real, estimado, s=3, alpha=0.4)
lim = [real.min(), real.max()]
plt.plot(lim, lim, 'r--')
plt.xlabel('Torque real (Nm)'); plt.ylabel('Torque predicho (Nm)'); plt.show()

### Extras (si sobra tiempo)
- Entrena de nuevo **sin estandarizar** los datos y compara. ¿Qué observas?
- Compara con `LinearRegression` de scikit-learn. ¿Justifica el problema usar una red?
- Entrena durante muchas más épocas. ¿Aparece sobreajuste (la pérdida de prueba sube mientras la de entrenamiento baja)?

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

lin = ...      # TODO: ajusta el modelo con X_tr, y_tr
mse_lin = ...  # TODO: MSE en prueba
print(f"MSE regresión lineal: {mse_lin:.4f}")

# DÍA 2

## 0. Configuración
Si reiniciaste el entorno, vuelve a ejecutar esta celda. Verifica que dice `cuda`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim

torch.manual_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Usando:", device)

## 7. Dimensiones en una CNN
**Reto:** antes de ejecutar nada, calcula a mano la forma de salida de cada capa para una imagen CIFAR-10 de $3\times32\times32$. Escribe tus predicciones y la celda te dirá si acertaste.

Para el `Flatten`, la forma es una tupla de un solo elemento, por ejemplo `(100,)`.

In [ ]:
parte_conv = nn.Sequential(
    nn.Conv2d(3, 16, kernel_size=5),    # capa 0
    nn.ReLU(),                          # capa 1
    nn.MaxPool2d(2),                    # capa 2
    nn.Conv2d(16, 32, kernel_size=3),   # capa 3
    nn.ReLU(),                          # capa 4
    nn.MaxPool2d(2),                    # capa 5
    nn.Flatten()                        # capa 6
)

In [ ]:
# TODO: escribe tu predicción (canales, alto, ancho) para cada capa
prediccion = [
    (..., ..., ...),   # capa 0
    (..., ..., ...),   # capa 1
    (..., ..., ...),   # capa 2
    (..., ..., ...),   # capa 3
    (..., ..., ...),   # capa 4
    (..., ..., ...),   # capa 5
    (...,),            # capa 6
]

In [ ]:
x = torch.zeros(1, 3, 32, 32)          # (batch, canales, alto, ancho)
for i, (capa, pred) in enumerate(zip(parte_conv, prediccion)):
    x = capa(x)
    real = tuple(x.shape[1:])
    estado = "OK" if pred == real else "NO"
    print(f"{estado:2s} capa {i} {capa.__class__.__name__:10s} real={real}  tu predicción={pred}")

Esta función cuenta los parámetros de un modelo. La usaremos más adelante.

In [ ]:
def contar_parametros(modelo):
    return sum(p.numel() for p in modelo.parameters())

print(contar_parametros(parte_conv))

**Reto extra:** ¿cuántos parámetros tiene `Conv2d(3, 16, kernel_size=5)`? Calcúlalo a mano y compáralo con el resultado de `contar_parametros(parte_conv[0])`.

## 8. CNN para PneumoniaMNIST
Radiografías de tórax de $28\times28$ en escala de grises. Clasificación binaria: normal vs. neumonía. Aplicamos la misma receta.

### Paso 1 · Datos

In [ ]:
!pip install -q medmnist
from medmnist import PneumoniaMNIST
from torchvision import transforms
from torch.utils.data import DataLoader

tf = transforms.ToTensor()      # tensor y escala a [0, 1]
train_ds = PneumoniaMNIST(split='train', download=True, transform=tf)
test_ds = PneumoniaMNIST(split='test', download=True, transform=tf)

train_dl = DataLoader(train_ds, batch_size=64, shuffle=True)
test_dl = DataLoader(test_ds, batch_size=64)

print(len(train_ds), len(test_ds))

In [ ]:
fig, axs = plt.subplots(1, 6, figsize=(12, 2.5))
for ax, i in zip(axs, range(6)):
    img, etiqueta = train_ds[i]
    ax.imshow(img.squeeze(), cmap='gray')
    ax.set_title(f"clase {int(etiqueta[0])}"); ax.axis('off')
plt.show()

print("Conteo por clase (train):", np.bincount(train_ds.labels.ravel()))
print("Conteo por clase (test): ", np.bincount(test_ds.labels.ravel()))

**Pregunta:** si un modelo predijera siempre la clase más frecuente, ¿qué precisión tendría en prueba? Calcúlalo; será tu referencia para juzgar la CNN.

In [ ]:
conteo = np.bincount(test_ds.labels.ravel())
precision_trivial = ...      # TODO
print(f"Precisión de un modelo trivial: {precision_trivial:.2%}")

### Paso 2 · Modelo

In [ ]:
modelo_neu = nn.Sequential(
    nn.Conv2d(1, 16, kernel_size=3, padding=1),    # 1x28x28  -> 16x28x28
    nn.ReLU(),
    nn.MaxPool2d(2),                               #          -> 16x14x14
    nn.Conv2d(16, 32, kernel_size=3, padding=1),   #          -> 32x14x14
    nn.ReLU(),
    nn.MaxPool2d(2),                               #          -> 32x7x7
    nn.Flatten(),                                  #          -> 1568
    nn.Linear(32 * 7 * 7, 64),
    nn.ReLU(),
    nn.Linear(64, 2)
)
print(contar_parametros(modelo_neu), "parámetros")

### Pasos 3, 4 y 5 · Pérdida, optimizador y bucle de entrenamiento

In [ ]:
modelo_neu = modelo_neu.to(device)
criterio = nn.CrossEntropyLoss()
optimizador = optim.Adam(modelo_neu.parameters(), lr=0.001)

epocas = 10
for epoca in range(epocas):
    modelo_neu.train()
    suma = 0
    for imgs, etiquetas in train_dl:
        imgs = imgs.to(device)
        etiquetas = etiquetas.squeeze().long().to(device)   # (N, 1) -> (N)

        optimizador.zero_grad()
        loss = criterio(modelo_neu(imgs), etiquetas)
        loss.backward()
        optimizador.step()
        suma += loss.item()
    print(f"Época {epoca+1} | Pérdida promedio: {suma/len(train_dl):.4f}")

### Evaluación
Estas dos funciones contienen el bucle que acabas de ver y las usaremos en CIFAR-10.

In [ ]:
def entrenar(modelo, dl, criterio, optimizador, epocas=10):
    perdidas = []
    for epoca in range(epocas):
        modelo.train()
        suma = 0
        for imgs, etiquetas in dl:
            imgs = imgs.to(device)
            etiquetas = etiquetas.squeeze().long().to(device)
            optimizador.zero_grad()
            loss = criterio(modelo(imgs), etiquetas)
            loss.backward()
            optimizador.step()
            suma += loss.item()
        perdidas.append(suma / len(dl))
        print(f"Época {epoca+1} | Pérdida promedio: {perdidas[-1]:.4f}")
    return perdidas


def evaluar(modelo, dl):
    modelo.eval()
    correctas, total = 0, 0
    with torch.no_grad():
        for imgs, etiquetas in dl:
            imgs = imgs.to(device)
            etiquetas = etiquetas.squeeze().long().to(device)
            pred = modelo(imgs).argmax(dim=1)
            correctas += (pred == etiquetas).sum().item()
            total += etiquetas.size(0)
    return correctas / total


print(f"Precisión en prueba: {evaluar(modelo_neu, test_dl):.2%}")

**Preguntas**
1. ¿Supera la CNN a la precisión del modelo trivial que calculaste?
2. En medicina, ¿basta con la precisión? ¿Qué error es más grave: un falso negativo o un falso positivo?

**Reto opcional:** compara con una red densa (sin convoluciones) con `Flatten` y `Linear`. Reporta precisión y número de parámetros de ambas.

In [ ]:
modelo_denso = nn.Sequential(
    nn.Flatten(),
    ...                         # TODO
).to(device)

criterio_d = ...
optimizador_d = ...
entrenar(modelo_denso, train_dl, criterio_d, optimizador_d, epocas=10)

print(f"Densa: {evaluar(modelo_denso, test_dl):.2%} con {contar_parametros(modelo_denso)} parámetros")
print(f"CNN:   {evaluar(modelo_neu, test_dl):.2%} con {contar_parametros(modelo_neu)} parámetros")

## 9. Ejercicio: CNN para CIFAR-10
**Objetivo:** adaptar la CNN de PneumoniaMNIST a imágenes a color de $3\times32\times32$ con 10 clases.

**Qué cambia respecto a PneumoniaMNIST**
- Número de canales de entrada.
- Tamaño de la entrada de la primera capa `Linear` (calcúlalo como en la sección 7).
- Número de clases.

### Paso 1 · Datos (ya está listo)

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

tf = transforms.ToTensor()
train_cifar = datasets.CIFAR10('./data', train=True, download=True, transform=tf)
test_cifar = datasets.CIFAR10('./data', train=False, download=True, transform=tf)
clases = train_cifar.classes
print(clases)

# Si el entrenamiento es muy lento, descomenta estas dos líneas para usar 10 000 imágenes:
# from torch.utils.data import Subset
# train_cifar = Subset(train_cifar, range(10000))

train_dl_c = DataLoader(train_cifar, batch_size=128, shuffle=True)
test_dl_c = DataLoader(test_cifar, batch_size=256)

In [ ]:
fig, axs = plt.subplots(2, 5, figsize=(10, 4.5))
for ax, i in zip(axs.ravel(), range(10)):
    img, etiqueta = test_cifar[i]
    ax.imshow(img.permute(1, 2, 0))
    ax.set_title(clases[etiqueta]); ax.axis('off')
plt.show()

### Paso 2 · Modelo
Completa los `...` y ejecuta la verificación de la celda siguiente: la salida debe tener forma `torch.Size([1, 10])`.

In [ ]:
modelo_cifar = nn.Sequential(
    nn.Conv2d(..., 16, kernel_size=3, padding=1),    # TODO: canales de entrada
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Conv2d(16, 32, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Flatten(),
    nn.Linear(..., 64),                              # TODO: entradas de la capa lineal
    nn.ReLU(),
    nn.Linear(64, ...)                               # TODO: número de clases
)

In [ ]:
x = torch.zeros(1, 3, 32, 32)
print(modelo_cifar(x).shape)     # esperado: torch.Size([1, 10])
print(contar_parametros(modelo_cifar), "parámetros")

Si falla con un error de dimensiones, el mensaje indica qué forma llegó a la capa `Linear`. Úsalo para corregir el número de entradas.

### Pasos 3, 4 y 5 · Pérdida, optimizador y entrenamiento
Usa la función `entrenar` de la sección 8.

In [ ]:
modelo_cifar = modelo_cifar.to(device)
criterio_c = ...          # TODO
optimizador_c = ...       # TODO

perdidas_c = entrenar(modelo_cifar, train_dl_c, criterio_c, optimizador_c, epocas=10)

### Evaluación

In [ ]:
plt.plot(perdidas_c); plt.xlabel('época'); plt.ylabel('pérdida'); plt.show()
print(f"Precisión en prueba: {evaluar(modelo_cifar, test_dl_c):.2%}")

**Preguntas**
1. ¿Qué precisión obtuviste? Un clasificador aleatorio tendría 10 %.
2. ¿Qué clases crees que se confunden más? ¿Por qué?

**Reto:** agrega una tercera capa convolucional (`Conv2d → ReLU → MaxPool2d`) y compara precisión y número de parámetros. Recuerda recalcular el tamaño de la capa `Linear` (usa la forma de la sección 7).

In [ ]:
modelo_cifar_2 = nn.Sequential(
    ...                    # TODO
).to(device)

print(modelo_cifar_2(torch.zeros(1, 3, 32, 32).to(device)).shape)   # esperado: [1, 10]

In [ ]:
criterio_c2 = ...
optimizador_c2 = ...
entrenar(modelo_cifar_2, train_dl_c, criterio_c2, optimizador_c2, epocas=10)

print(f"2 capas conv: {evaluar(modelo_cifar, test_dl_c):.2%} ({contar_parametros(modelo_cifar)} parámetros)")
print(f"3 capas conv: {evaluar(modelo_cifar_2, test_dl_c):.2%} ({contar_parametros(modelo_cifar_2)} parámetros)")

# Cierre
Revisa que puedes explicar, con tus palabras, cada paso de la receta:

1. **Datos:** ¿qué forma tienen los tensores de entrada y de etiquetas?
2. **Modelo:** ¿cuántas entradas y salidas tiene la red y por qué?
3. **Pérdida:** ¿`MSELoss` o `CrossEntropyLoss`? ¿Por qué?
4. **Optimizador:** ¿qué hace `lr`?
5. **Bucle:** `zero_grad → forward → loss → backward → step`.

¿Qué cambió entre *moons*, AI4I, PneumoniaMNIST y CIFAR-10? ¿Qué se mantuvo igual?